In [2]:
# Import the basic spark library
from pyspark.sql import SparkSession

# Create an entry point to the PySpark Application
spark = SparkSession.builder \
      .master("local") \
      .appName("ProjSMBUD") \
      .getOrCreate()
# master contains the URL of your remote spark instance or 'local'

In [3]:
# Load DataFrames (we prefer directly them to the rdd, because we already had csv with headers)
#Configuring them with some examples to check the correctness (now commented before and after the casting)

df_incollection = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ";")\
                .csv("incollectionFormatted.csv.txt")
df_book = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ";")\
                .csv("bookFormatted.csv.txt")
df_author = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ",")\
                .csv("authorFormatted.csv.txt")


# We use of datatypes from pyspark.sql.types
from pyspark.sql.types import StringType, DateType, FloatType, StructType, StructField, ArrayType
from pyspark.sql.functions import * 


##We use the ids in string format!
##TO HAVE THE ARRAY OF STRINGS
df_book = df_book\
    .select("id","date", "url", "isbn", "citationscore", "publisher" , "title", split(col("editor_array"),",").alias("editor_array"))
df_incollection = df_incollection\
    .select("id","title","date", "url", "pages", "citationscore", \
    "book_id", split(col("author_array"),",").alias("author_array"), \
    split(col("cited_inc_array"),",").alias("cited_inc_array"))

#LOAD INCOLLECTIONS
df_incollection = df_incollection \
  .withColumn("id" ,
              df_incollection["id"]
              .cast(StringType()))   \
  .withColumn("title"  ,
              df_incollection["title"]
              .cast(StringType())) \
  .withColumn("date",
              df_incollection["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_incollection["url"]
              .cast(StringType())) \
  .withColumn("pages"  ,
              df_incollection["pages"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_incollection["citationscore"]
              .cast(FloatType()))

#LOAD BOOKS
df_book = df_book \
  .withColumn("id" ,
              df_book["id"]
              .cast(StringType()))   \
  .withColumn("isbn"  ,
              df_book["isbn"]
              .cast(StringType())) \
  .withColumn("date",
              df_book["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_book["url"]
              .cast(StringType())) \
  .withColumn("publisher"  ,
              df_book["publisher"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_book["citationscore"]
              .cast(FloatType())) \
  .withColumn("title"  ,
              df_book["title"]
              .cast(StringType()))

#LOAD AUTHORS
df_author = df_author \
  .withColumn("id" ,
              df_author["id"]
              .cast(StringType()))   \
  .withColumn("name"  ,
              df_author["name"]
              .cast(StringType())) \
  .withColumn("citationscore",
              df_author["citationscore"]
              .cast(FloatType()))



In [4]:
df_book.printSchema()
df_incollection.printSchema()
df_author.printSchema()

root
 |-- id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- url: string (nullable = true)
 |-- isbn: string (nullable = true)
 |-- citationscore: float (nullable = true)
 |-- publisher: string (nullable = true)
 |-- title: string (nullable = true)
 |-- editor_array: array (nullable = true)
 |    |-- element: string (containsNull = false)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date: date (nullable = true)
 |-- url: string (nullable = true)
 |-- pages: string (nullable = true)
 |-- citationscore: float (nullable = true)
 |-- book_id: string (nullable = true)
 |-- author_array: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- cited_inc_array: array (nullable = true)
 |    |-- element: string (containsNull = false)

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- citationscore: float (nullable = true)
 |-- email: string (nullable = true)



<b>Query 1:</b> <br>Find the titles of incollections written after 01/01/2015 by the author "Yvo Desmedt" together with at least another author.

<i>Complexity: WHERE, IN, Nested Query</i>

In [5]:
from pyspark.sql.functions import explode, col, array_contains

author_id = df_author.filter(col("name") == "Yvo Desmedt")\
                     .select(col("id"))\
                     .collect()

author_id = author_id[0][0]

df_incollection.filter(col("date") > lit("2015-01-01"))\
               .filter(array_contains(col("author_array"), author_id))\
               .filter(size("author_array") > 1)\
               .select(col("title"),col("date"),col("author_array"))\
               .show(truncate = False)

+-------------------------------------------------------------------+----------+------------+
|title                                                              |date      |author_array|
+-------------------------------------------------------------------+----------+------------+
|Perfectly Secure Message Transmission.                             |2017-07-12|[7509, 7387]|
|Broadcast Authentication from an Information Theoretic Perspective.|2017-07-12|[7648, 7387]|
|Stream and Multicast Authentication.                               |2017-07-12|[7648, 7387]|
+-------------------------------------------------------------------+----------+------------+



<b>Query 2:</b><br>
Finds the Ids of the best 10 books with the following steps: 
<ul>
<li>Select the incollections which are cited at least by 25 other incollections (it is supposed that an incollection is important if it's cited a large number of times by other pubblications).</li>

<li>Count for each book how many of the incollections selected with the previous query are present in it.</li>
    
<li>Order the books using the calculated number and limit to 10 the query result.</li>
</ul>
<i>Complexity: WHERE, Nested Query, GROUP BY </i>

In [7]:
from pyspark.sql.functions import explode, col

# for each incollection explode the cited_inc_array, it is obtained one column of ids
exploded_df = df_incollection.select(explode(df_incollection.cited_inc_array))\
                             .withColumnRenamed("col", "cite")

#count how many times each incollection appears and discard the incollections cited less than 25 times 
cited_ids = exploded_df.groupBy("cite")\
                       .count()\
                       .filter(col("count") > 25)\
                       .select("cite")\
                       .collect()

#convert the dataframe to an array
cited_ids = [ci[0] for ci in cited_ids]

#find the books which contains the incollections and count the number for each book
df_incollection.filter(col("id").isin(cited_ids))\
               .groupBy("book_id")\
               .count()\
               .orderBy(col("count").desc())\
               .limit(10)\
               .show()

+-------+-----+
|book_id|count|
+-------+-----+
|   5081|    7|
|   5067|    7|
|   5150|    7|
|   5126|    7|
|   5143|    7|
|   5110|    6|
|   5128|    5|
|   5145|    5|
|   5177|    5|
|   5132|    5|
+-------+-----+



<b>Command 1:</b> <br>Create a new incollection with id 10

In [82]:
from datetime import datetime

schema = StructType([ \
    StructField("id", StringType(), True), \
    StructField("title", StringType(), True), \
    StructField("date", DateType(), True), \
    StructField("url", StringType(), True), \
    StructField("pages", StringType(), True), \
    StructField("citationscore", FloatType(), True), \
    StructField("book_id", StringType(), True), \
    StructField("author_array", ArrayType(StringType()), True), \
    StructField("cited_inc_array", ArrayType(StringType()), True) \
])

new_incollection= [("10", "Efficient View Maintenance at Data Warehouses.", datetime.strptime("2022-09-09","%Y-%m-%d"),"http://dblp.uni-trier.de/papers/DiplomarbeitRitaLey.pdf", "456-345", 0.962399934, "5110",["2008","2128"],["5409","5410","6543"])]


df_new_inc = spark.createDataFrame(data = new_incollection, schema = schema)
df_new_inc.show()

df_incollection = df_incollection.union(df_new_inc)

+---+--------------------+----------+--------------------+-------+-------------+-------+------------+------------------+
| id|               title|      date|                 url|  pages|citationscore|book_id|author_array|   cited_inc_array|
+---+--------------------+----------+--------------------+-------+-------------+-------+------------+------------------+
| 10|Efficient View Ma...|2022-09-09|http://dblp.uni-t...|456-345|   0.96239996|   5110|[2008, 2128]|[5409, 5410, 6543]|
+---+--------------------+----------+--------------------+-------+-------------+-------+------------+------------------+



Check for the new insertion

In [90]:
df_incollection.orderBy(col("id").asc()).limit(5).show()

+----+--------------------+----------+--------------------+---------+-------------+-------+------------+--------------------+
|  id|               title|      date|                 url|    pages|citationscore|book_id|author_array|     cited_inc_array|
+----+--------------------+----------+--------------------+---------+-------------+-------+------------+--------------------+
|  10|Efficient View Ma...|2022-09-09|http://dblp.uni-t...|  456-345|   0.96239996|   5110|[2008, 2128]|  [5409, 5410, 6543]|
|5367|Ear Shape for Bio...|2017-07-12|https://doi.org/1...|  372-378|    0.8379688|   5169|      [7367]|[6264, 7146, 7146...|
|5368|      ARP Poisoning.|2017-07-12|https://doi.org/1...|       47|    0.9060627|   5067|        null|[5480, 6390, 6371...|
|5369|              Token.|2017-07-12|https://doi.org/1...|1305-1306|    0.8996254|   5109|      [7368]|[5946, 5866, 6565...|
|5370|Multi-Threaded Im...|2017-07-12|https://doi.org/1...|  823-824|    0.6736637|   5114|      [7369]|[7056, 7059, 7